[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eygpcr/biyofizik2026-martini/blob/main/notebooks/04_martini_input.ipynb)

# Oturum 4 — Martini 3 Girdi Hazırlama

**Biyofizik 2026 Kursu · 25 Ağustos 2026 · Dr. Öğr. Üyesi Ekrem Yaşar**

Sabah oturumlarında CHARMM-GUI arayüzü ile atomistik olarak hazırlanan protein
(`6JOD` A zinciri, anjiyotensin II tip-2 reseptörü), bu not defterinde komut
satırı araçlarıyla ve Martini 3 kaba-taneli modeli kullanılarak yeniden
hazırlanmaktadır.

Kursta üretim simülasyonu koşulmamaktadır; amaç simülasyona girecek sistemin
kurulmasıdır.

Her adımın sonunda üretilen yapı **etkileşimli olarak görüntülenmekte** ve
statik bir çizimi Google Drive'a kaydedilmektedir.

**İşlem sırası**

| Bölüm | Konu |
|---|---|
| 1 | Google Drive bağlanması ve çalışma klasörü |
| 2 | Yazılım kurulumu |
| 3 | Görselleştirme araçları |
| 4 | Yapının indirilmesi ve incelenmesi |
| 5 | A zincirinin ayıklanması |
| 6 | İkincil yapının belirlenmesi |
| 7 | `martinize2` ile kaba-taneli modele dönüştürme |
| 8 | Martini 3 kuvvet alanı dosyaları |
| 9 | `insane` ile membran, çözücü ve iyon eklenmesi |
| 10 | İyon adlarının düzeltilmesi |
| 11 | Topolojinin tamamlanması |
| 12 | `gmx grompp` ile doğrulama |
| 13 | Sistemin analizi ve görselleştirilmesi |
| 14 | Drive klasörünün özeti |


---
## 1. Google Drive bağlanması

Colab çalışma zamanı sonlandığında üretilen dosyalar silinmektedir. Bu nedenle
çıktılar Google Drive üzerinde kalıcı bir klasöre kaydedilecektir.

Aşağıdaki hücre çalıştırıldığında Google hesabına erişim izni istenecektir;
açılan pencerede kendi hesabınızı seçip izin veriniz.

```
Drive'ım/
└── Biyofizik2026_Martini/
    └── martini_input/          <- bu oturum
        ├── girdi/              ham ve hazırlanmış yapılar
        ├── cikti/              koordinat, topoloji ve parametre dosyaları
        └── gorseller/          çizimler
```

**Not.** Oturum 6 aynı `Biyofizik2026_Martini` klasörü altında ayrı bir
`bentopy/` alt klasörü kullanmaktadır; oturumların çıktıları karışmamaktadır.


In [ ]:
from google.colab import drive
import os, shutil, glob

drive.mount('/content/drive')

# --- Kalici klasor yapisi (Google Drive) ---
DRIVE_KOK  = '/content/drive/MyDrive/Biyofizik2026_Martini'
OTURUM     = os.path.join(DRIVE_KOK, 'martini_input')
D_GIRDI    = os.path.join(OTURUM, 'girdi')
D_CIKTI    = os.path.join(OTURUM, 'cikti')
D_GORSEL   = os.path.join(OTURUM, 'gorseller')

for d in (DRIVE_KOK, OTURUM, D_GIRDI, D_CIKTI, D_GORSEL):
    os.makedirs(d, exist_ok=True)

# --- Hizli yerel calisma dizini ---
# Agir dosya islemleri Drive uzerinde yavas oldugundan yerelde calisilir,
# uretilen dosyalar adim adim Drive'a kopyalanir.
CALISMA = '/content/calisma'
os.makedirs(CALISMA, exist_ok=True)
os.chdir(CALISMA)

print('Drive klasoru :', OTURUM)
print('Calisma dizini:', os.getcwd())


In [ ]:
def kaydet(desenler, hedef, sessiz=False):
    """Verilen dosya desenlerini Drive'daki hedef klasore kopyalar."""
    kopyalanan = []
    for desen in desenler:
        for dosya in glob.glob(desen):
            if os.path.isfile(dosya):
                shutil.copy(dosya, hedef)
                kopyalanan.append(os.path.basename(dosya))
    if not sessiz:
        if kopyalanan:
            print(f'Drive\'a kaydedildi ({os.path.basename(hedef)}/):',
                  ', '.join(sorted(kopyalanan)))
        else:
            print('Kopyalanacak dosya bulunamadi:', desenler)
    return kopyalanan


---
## 2. Yazılım kurulumu

Aşağıdaki hücrenin çalışma süresi yaklaşık 3–5 dakikadır ve oturum başına bir
kez çalıştırılması yeterlidir.

| Paket | İşlevi |
|---|---|
| `vermouth` | `martinize2` komutunu sağlamaktadır |
| `insane` | Membran ve kutu inşası |
| `gromacs` | `gmx grompp` ile doğrulama |
| `py3Dmol` | Yapıların not defteri içinde görüntülenmesi |


In [ ]:
%%capture
!pip install -q vermouth insane py3Dmol
!apt-get -qq update
!apt-get -qq install -y gromacs


In [ ]:
# Kurulumun dogrulanmasi
!martinize2 --version 2>&1 | head -2
!insane --help 2>&1 | head -3
!gmx --version 2>&1 | grep -i 'GROMACS version'
import py3Dmol; print('py3Dmol hazir')


---
## 3. Görselleştirme araçları

Bu bölümde tanımlanan iki fonksiyon not defteri boyunca kullanılacaktır:

| Fonksiyon | İşlevi |
|---|---|
| `yapi_goster()` | Yapıyı not defteri içinde **etkileşimli** olarak gösterir (döndürülebilir, yakınlaştırılabilir) |
| `kesit_ciz()` | Yapının **statik** bir kesit çizimini üretir ve Drive'a PNG olarak kaydeder |

Etkileşimli görünüm oturum sırasında incelemek için, statik çizim ise kalıcı
kayıt ve rapor için kullanılmaktadır.


In [ ]:
import py3Dmol
import numpy as np
import matplotlib.pyplot as plt

# --- Bilesen siniflandirmasi ---
LIPIT  = {'POPC','POPE','POPS','POPG','CHOL','DOPC','DPPC','DOPE'}
COZUCU = {'W','WF','PW'}
IYON   = {'NA','CL','NA+','CL-','ION','K','K+'}

RENK = {'Protein': '#2E5FA3', 'Lipit': '#E08A2E',
        'Cozucu': '#9BC49B', 'Iyon': '#C0392B'}


def yapi_goster(dosya, stil='oto', genislik=800, yukseklik=500,
                zincir_renkleri=None, cozucu_gizle=True):
    """Yapiyi not defteri icinde etkilesimli olarak gosterir."""
    bicim = 'gro' if dosya.endswith('.gro') else 'pdb'
    v = py3Dmol.view(width=genislik, height=yukseklik)
    v.addModel(open(dosya).read(), bicim)

    if zincir_renkleri:
        v.setStyle({}, {'cartoon': {'color': 'white'}})
        for zincir, renk in zincir_renkleri.items():
            v.setStyle({'chain': zincir}, {'cartoon': {'color': renk}})
    elif stil == 'karton':
        v.setStyle({}, {'cartoon': {'color': 'spectrum'}})
    elif stil == 'kure':
        v.setStyle({}, {'sphere': {'radius': 1.6}})
    elif stil == 'cg_protein':
        v.setStyle({}, {'sphere': {'radius': 1.8, 'color': '#7F8C8D'}})
        v.setStyle({'atom': 'BB'}, {'sphere': {'radius': 2.4, 'color': '#2E5FA3'}})
    elif stil == 'sistem':
        v.setStyle({}, {})
        if not cozucu_gizle:
            v.addStyle({'resn': list(COZUCU)},
                       {'sphere': {'radius': 0.7, 'color': '#BBD8BB', 'opacity': 0.35}})
        v.addStyle({'resn': list(LIPIT)},
                   {'sphere': {'radius': 1.2, 'color': '#E08A2E', 'opacity': 0.65}})
        v.addStyle({'resn': list(IYON)},
                   {'sphere': {'radius': 1.6, 'color': '#C0392B'}})
        # Protein: lipit/cozucu/iyon disinda kalan her sey
        v.addStyle({'resn': list(LIPIT | COZUCU | IYON), 'invert': True},
                   {'sphere': {'radius': 2.2, 'color': '#2E5FA3'}})
    else:
        v.setStyle({}, {'cartoon': {'color': 'spectrum'}})

    v.zoomTo()
    v.setBackgroundColor('white')
    return v.show()


def koordinat_oku(dosya, max_atom=400_000):
    """PDB veya GRO dosyasindan rezidu adi, atom adi ve koordinat (nm) okur."""
    if dosya.endswith('.gro'):
        satirlar = open(dosya).read().splitlines()
        n = int(satirlar[1])
        atomlar = satirlar[2:2+n]
        if n > max_atom:
            atomlar = atomlar[::(n // max_atom + 1)]
        res  = np.array([s[5:10].strip()  for s in atomlar])
        ad   = np.array([s[10:15].strip() for s in atomlar])
        xyz  = np.array([[float(s[20:28]), float(s[28:36]), float(s[36:44])]
                         for s in atomlar])
    else:
        satirlar = [l for l in open(dosya) if l.startswith(('ATOM  ', 'HETATM'))]
        res = np.array([l[17:20].strip() for l in satirlar])
        ad  = np.array([l[12:16].strip() for l in satirlar])
        xyz = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                        for l in satirlar]) / 10.0   # A -> nm
    return res, ad, xyz


def bilesen_maskeleri(res):
    """Rezidu adlarindan bilesen maskeleri uretir."""
    diger = list(LIPIT | COZUCU | IYON)
    return {
        'Protein': ~np.isin(res, diger),
        'Lipit'  : np.isin(res, list(LIPIT)),
        'Cozucu' : np.isin(res, list(COZUCU)),
        'Iyon'   : np.isin(res, list(IYON)),
    }


def kesit_ciz(dosya, png, baslik, dilim=None, eksen=('x', 'z')):
    """Yapinin yandan gorunumunu (izdusum) cizer ve PNG olarak kaydeder."""
    res, ad, xyz = koordinat_oku(dosya)
    i, j = {'x': 0, 'y': 1, 'z': 2}[eksen[0]], {'x': 0, 'y': 1, 'z': 2}[eksen[1]]
    k = 3 - i - j   # dilim alinacak eksen

    sec = np.ones(len(res), dtype=bool)
    if dilim is not None:
        orta = (xyz[:, k].min() + xyz[:, k].max()) / 2
        sec = np.abs(xyz[:, k] - orta) < dilim / 2

    maskeler = bilesen_maskeleri(res)
    plt.figure(figsize=(7.5, 7.5))
    cizim_sirasi = [('Cozucu', 1.6, 0.22), ('Lipit', 3.0, 0.60),
                    ('Iyon', 8.0, 0.85), ('Protein', 6.0, 0.95)]
    for ad_g, boyut, saydam in cizim_sirasi:
        m = maskeler[ad_g] & sec
        if m.sum() == 0:
            continue
        plt.scatter(xyz[m, i], xyz[m, j], s=boyut, c=RENK[ad_g],
                    alpha=saydam, linewidths=0,
                    label=f'{ad_g} ({m.sum():,} / {maskeler[ad_g].sum():,})')
    plt.xlabel(f'{eksen[0]} (nm)'); plt.ylabel(f'{eksen[1]} (nm)')
    plt.title(baslik)
    plt.gca().set_aspect('equal')
    plt.legend(loc='upper right', framealpha=.9, markerscale=4,
               title='dilimde / toplam', title_fontsize=8, fontsize=8)
    plt.grid(alpha=.2)
    plt.tight_layout()
    plt.savefig(png, dpi=150)
    plt.show()
    print('Kaydedildi:', png)

print('Gorsellestirme fonksiyonlari hazir.')


---
## 4. Yapının indirilmesi ve incelenmesi

`6JOD` yapısı beş zincir içermektedir. Aşağıdaki görünümde renkler şu anlama
gelmektedir:

| Renk | Zincir | Tanım | İşlem |
|---|---|---|---|
| Kırmızı | A | Anjiyotensin II tip-2 reseptörü (AT2R) | **korunur** |
| Sarı | B | Anjiyotensin II — agonist peptit | korunur |
| Gri | C | BRIL (çözünür sitokrom b562) — kristalizasyon füzyonu | **çıkarılır** |
| Açık mavi | H | 4A03 Fab ağır zincir | **çıkarılır** |
| Açık yeşil | L | 4A03 Fab hafif zincir | **çıkarılır** |

Görünümü döndürerek yapının ne kadarının deneysel yardımcı bileşenlerden
oluştuğunu inceleyiniz.


In [ ]:
!wget -q https://files.rcsb.org/download/6JOD.pdb -O 6jod.pdb
!ls -lh 6jod.pdb


In [ ]:
yapi_goster('6jod.pdb', zincir_renkleri={
    'A': 'red',        # AT2R          -> korunur
    'B': 'yellow',     # Anjiyotensin II -> korunur
    'C': 'grey',       # BRIL          -> cikarilir
    'H': 'lightblue',  # Fab agir      -> cikarilir
    'L': 'lightgreen', # Fab hafif     -> cikarilir
})


**Kavramsal not.** GPCR ailesine ait proteinler kristalizasyona dirençlidir.
Bu güçlüğün aşılması için proteine BRIL veya T4 lizozim gibi bir füzyon bölgesi
eklenmekte ya da yapı bir Fab fragmanı ile kompleks hâlinde kristalize
edilmektedir. Her iki bileşen de deneysel araçtır; fizyolojik ortamda
bulunmamaktadır.

Bu yapıda BRIL ayrı bir zincir olarak deposit edildiğinden çıkarılması doğrudan
zincir silme işlemiyle yapılabilmektedir. Bazı GPCR yapılarında BRIL üçüncü
hücre içi ilmiğin (ICL3) **içine** yerleştirilmiş olduğundan dizinin ortasından
kesilerek çıkarılması gerekir; bunun yapının kaynak makalesine başvurulmadan
anlaşılması mümkün değildir.


---
## 5. A zincirinin ayıklanması

Bu oturumda ligant (B zinciri) da alınmamakta, yalnızca protein ve membrandan
oluşan sade bir sistem kurulmaktadır.


In [ ]:
# Yalnizca A zincirinin ATOM kayitlari (su ve hetero gruplar haric)
kept = [l for l in open('6jod.pdb') if l.startswith('ATOM  ') and l[21] == 'A']

# DSSP ve bazi araclar CRYST1 kaydi bekledigi icin eklenir
cryst = 'CRYST1    1.000    1.000    1.000  90.00  90.00  90.00 P 1           1\n'
open('at2r.pdb', 'w').writelines([cryst] + kept + ['END\n'])

resids = sorted({int(l[22:26]) for l in kept})
print(f'A zinciri: {len(resids)} rezidu ({resids[0]}-{resids[-1]}), {len(kept)} atom')

kaydet(['6jod.pdb', 'at2r.pdb'], D_GIRDI)


**Beklenen sonuç.** 35–340 aralığında 306 rezidü.
Dizide 312 rezidü bulunmakta olup 341–346 aralığındaki C-terminal uzantı
çözülmemiştir. Zincir içi kopukluk bulunmadığından ilmik modellemesine gerek
duyulmamaktadır.


In [ ]:
yapi_goster('at2r.pdb', stil='karton')


Yalnızca reseptör kaldı. Renkler N-ucundan (mavi) C-ucuna (kırmızı) doğru
değişmektedir; yedi transmembran heliksi bu görünümde ayırt edilebilmektedir.


In [ ]:
kesit_ciz('at2r.pdb', os.path.join(D_GORSEL, '01_at2r_atomistik.png'),
          'Adim 1 - AT2R atomistik yapi (6JOD A zinciri)')


---
## 6. İkincil yapının belirlenmesi

Martini 3'te etkileşim merkezlerinin tipleri ikincil yapıya bağlı olduğundan
`martinize2` her rezidü için bir ikincil yapı ataması gerektirmektedir.

**`-dssp` seçeneği neden kullanılmıyor?** Ubuntu depolarındaki güncel `mkdssp`
sürümü (4.x) girdi dosyasında `CRYST1` kaydı aramakta, `vermouth` tarafından
üretilen geçici dosyada ise bu kayıt bulunmamaktadır. Bu nedenle `-dssp`
kullanımı `DSSPError: Expected record CRYST1 but found ATOM` hatası vermektedir.

**Kullanılan yöntem.** İkincil yapı, kristal yapının kendi `HELIX` ve `SHEET`
kayıtlarından çıkarılmakta ve `-ss` seçeneği ile doğrudan verilmektedir. Bu
atama, yapıyı çözen araştırmacılar tarafından yapılmış olduğundan güvenilirdir.

> Deneysel yapı yerine bir model (örneğin AlphaFold çıktısı) kullanılıyorsa bu
> kayıtlar bulunmayacaktır; o durumda uyumlu bir DSSP sürümü kurulmalı veya
> `mdtraj` gibi bir kütüphaneyle ikincil yapı hesaplanmalıdır.


In [ ]:
def ss_cikar(pdb, zincir='A'):
    """PDB dosyasinin HELIX/SHEET kayitlarindan ikincil yapi dizesi uretir."""
    resids = sorted({int(l[22:26]) for l in open(pdb)
                     if l.startswith('ATOM  ') and l[21] == zincir})
    ss = {r: 'C' for r in resids}
    nh = ne = 0
    for l in open(pdb):
        if l.startswith('HELIX') and l[19] == zincir:
            nh += 1
            for r in range(int(l[21:25]), int(l[33:37]) + 1):
                if r in ss: ss[r] = 'H'
        elif l.startswith('SHEET') and l[21] == zincir:
            ne += 1
            for r in range(int(l[22:26]), int(l[33:37]) + 1):
                if r in ss: ss[r] = 'E'
    dize = ''.join(ss[r] for r in resids)
    print(f'HELIX kaydi: {nh} | SHEET kaydi: {ne}')
    print(f'Dize uzunlugu: {len(dize)} (rezidu sayisi: {len(resids)})')
    print(f'Dagilim -> H: {dize.count("H")} (%{100*dize.count("H")/len(dize):.0f})  '
          f'E: {dize.count("E")}  C: {dize.count("C")}')
    return dize

SS = ss_cikar('6jod.pdb', 'A')
print()
for i in range(0, len(SS), 60):
    print(f'{i+1:>4}  {SS[i:i+60]}')


**Beklenen sonuç.** 306 karakterlik bir dize; heliks oranı yaklaşık %84.
Yedi transmembran heliksi ve hücre dışı ikinci ilmikteki (ECL2) kısa β-firkete
(`E` bloğu) bu dizede görülebilmektedir. Bu dağılım bir GPCR için beklenen
yapıya uymaktadır.


---
## 7. `martinize2` ile kaba-taneli modele dönüştürme

| Parametre | İşlevi | Önemi |
|---|---|---|
| `-ff martini3001` | Martini 3 kuvvet alanı | Martini 2 ile karıştırılmamalıdır |
| `-ss` | İkincil yapı ataması | Merkez tipleri ikincil yapıya bağlıdır |
| `-elastic` | Elastik ağ tanımlanması | Uygulanmadığında yapısal bütünlük korunamamaktadır |
| `-ef 700` | Yay kuvvet sabiti (kJ mol⁻¹ nm⁻²) | Yüksek değer aşırı rijitlik, düşük değer yapısal bozulma |
| `-el 0.5 -eu 0.9` | Yay mesafe aralığı (nm) | Bağlanacak merkez çiftlerini belirlemektedir |
| `-cys auto` | Disülfit köprülerinin otomatik belirlenmesi | Cys35–Cys290 ve Cys117–Cys195 |


In [ ]:
!martinize2 \
  -f at2r.pdb \
  -o topol.top \
  -x at2r_cg.pdb \
  -ff martini3001 \
  -ss {SS} \
  -elastic -ef 700 -el 0.5 -eu 0.9 -ea 0 -ep 0 \
  -cys auto \
  -maxwarn 10


Çıktıdaki `Disulfide bridge found between residues A-CYS1 and A-CYS256` ile
`A-CYS83 and A-CYS161` iletileri, `martinize2`'nin kendi iç numaralandırmasını
kullanmaktadır. PDB numaralandırmasına çevrildiğinde bunlar **Cys35–Cys290** ve
**Cys117–Cys195** köprülerine karşılık gelmektedir; ikisi de yapının `SSBOND`
kayıtlarıyla uyumludur.


In [ ]:
aa = sum(1 for l in open('at2r.pdb')    if l.startswith('ATOM'))
cg = sum(1 for l in open('at2r_cg.pdb') if l.startswith(('ATOM','HETATM')))
print(f'Atomistik model  : {aa:>6} atom')
print(f'Kaba-taneli model: {cg:>6} etkilesim merkezi')
print(f'Indirgeme orani  : {aa/cg:.1f}')
print()
kaydet(['at2r_cg.pdb', 'molecule_*.itp', 'topol.top'], D_CIKTI)


### Kaba-taneli yapının görünümü

Mavi küreler omurga (BB) merkezlerini, gri küreler yan zincir (SC) merkezlerini
göstermektedir. Atomistik görünümle karşılaştırınız: molekülün genel şekli
korunmakta, ancak atomik ayrıntı kaybolmaktadır.


In [ ]:
yapi_goster('at2r_cg.pdb', stil='cg_protein')


In [ ]:
# Atomistik ve kaba-taneli modelin yan yana karsilastirmasi
res_aa, ad_aa, xyz_aa = koordinat_oku('at2r.pdb')
res_cg, ad_cg, xyz_cg = koordinat_oku('at2r_cg.pdb')

fig, eksenler = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
for eks, (xyz, ad_g, renk, boyut) in zip(eksenler, [
        (xyz_aa, f'Atomistik ({len(xyz_aa):,} atom)', '#2E5FA3', 4),
        (xyz_cg, f'Martini 3 ({len(xyz_cg):,} merkez)', '#C0392B', 22)]):
    eks.scatter(xyz[:, 0], xyz[:, 2], s=boyut, c=renk, alpha=.6, linewidths=0)
    eks.set_title(ad_g); eks.set_xlabel('x (nm)'); eks.set_aspect('equal')
    eks.grid(alpha=.2)
eksenler[0].set_ylabel('z (nm)')
fig.suptitle('Adim 2 - cozunurluk indirgeme', fontsize=13)
plt.tight_layout()
png = os.path.join(D_GORSEL, '02_atomistik_vs_kaba_taneli.png')
plt.savefig(png, dpi=150); plt.show()
print('Kaydedildi:', png)


**Elastik ağın gerekliliği.** Martini kuvvet alanı, etkileşim merkezleri
arasındaki potansiyeller aracılığıyla proteinin üçüncül yapısını
koruyamamaktadır. Yapı, harmonik yaylardan oluşan bir ağ ile kısıtlanmaktadır.

Bu yaklaşımın sonucu olarak model protein katlanma, açılma ve büyük ölçekli
konformasyonel değişim gösterememektedir. Dolayısıyla Martini modeli ile bir
GPCR'ın aktivasyon geçişi incelenememektedir. İncelenebilecek süreçler lipit
etkileşimleri, oligomerizasyon ve difüzyondur.


In [ ]:
# Uretilen topolojinin yapisi
itp = sorted(glob.glob('molecule_*.itp'))[0]
satirlar = open(itp).read().splitlines()
print('Dosya:', itp, '|', len(satirlar), 'satir\n')
bolum = None; sayim = {}
for l in satirlar:
    s = l.strip()
    if s.startswith('['):
        bolum = s.strip('[] ')
        sayim[bolum] = 0
    elif s and not s.startswith(';') and bolum:
        sayim[bolum] += 1
for b, n in sayim.items():
    print(f'  {b:<18} {n:>6} satir')


---
## 8. Martini 3 kuvvet alanı dosyaları

`martinize2` proteinin topolojisini üretmiştir. Ayrıca Martini kuvvet alanının
genel parametre dosyaları (etkileşim merkezi tanımları, lipitler, çözücü ve
iyonlar) gerekmektedir.

Kaynak: [marrink-lab/martini-forcefields](https://github.com/marrink-lab/martini-forcefields)

Ana parametre dosyasının boyutu yaklaşık 16 MB olup indirme süresi bir dakikayı
bulabilmektedir. Bu dosyalar her oturumda yeniden indirilebildiğinden Drive'a
kaydedilmemektedir.


In [ ]:
BASE = 'https://raw.githubusercontent.com/marrink-lab/martini-forcefields/main/martini_forcefields/regular/v3.0.0/gmx_files'
for f in ['martini_v3.0.0.itp',
          'martini_v3.0.0_solvents_v1.itp',
          'martini_v3.0.0_ions_v1.itp',
          'martini_v3.0.0_phospholipids_v1.itp']:
    !wget -q {BASE}/{f} -O {f}
!ls -lh martini_v3.0.0*.itp

# Iyonlarin kuvvet alanindaki adlari (10. bolumde gerekecek)
print('\nIyon molecule tipleri:')
!grep -A2 'moleculetype' martini_v3.0.0_ions_v1.itp | grep -E '^(NA|CL|K|CA) ' | head


---
## 9. `insane` ile membran, çözücü ve iyon eklenmesi

| Parametre | İşlevi |
|---|---|
| `-box 12,12,14` | Kutu boyutları (nm) |
| `-l POPC:1` | Lipit bileşimi |
| `-sol W` | Martini standart su modeli (bir merkez yaklaşık dört su molekülü) |
| `-salt 0.15` | 0.15 M NaCl |
| `-center` | Proteinin kutuya ortalanması |
| `-dm 0` | Proteinin membran düzlemine göre z ötelemesi |

Çok bileşenli membran için: `-l POPC:7 -l POPE:2 -l CHOL:1`


In [ ]:
!insane \
  -f at2r_cg.pdb \
  -o sistem_ham.gro \
  -p sistem_insane.top \
  -pbc square \
  -box 12,12,14 \
  -l POPC:1 \
  -sol W \
  -salt 0.15 \
  -center \
  -dm 0


In [ ]:
print('--- insane tarafindan uretilen topoloji ---')
print(open('sistem_insane.top').read())
print('Toplam parcacik sayisi:', f"{int(open('sistem_ham.gro').read().splitlines()[1]):,}")


---
## 10. İyon adlarının düzeltilmesi

**Bu adım atlanırsa `gmx grompp` şu hatayı verir:**

```
ERROR 1 [file sistem.top]:
  No such moleculetype NA+
```

**Nedeni.** `insane` aracı Martini 2 döneminde geliştirilmiş olup iyonları
`NA+` ve `CL-` adlarıyla üretmektedir. Martini 3 kuvvet alanı ise aynı iyonları
`NA` ve `CL` adlarıyla tanımlamaktadır (8. bölümdeki çıktıda görülebilir).
Adlar eşleşmediğinden GROMACS molekül tipini bulamamaktadır.

**Çözüm.** Hem koordinat dosyasındaki hem topolojideki adların dönüştürülmesi.

> Bu, `insane` ile Martini 3'ün birlikte kullanımında karşılaşılan en yaygın
> sorundur. Kendi sistemlerinizi kurarken de aynı düzeltmeyi yapmanız
> gerekecektir.


In [ ]:
ESLEME = {'NA+': 'NA', 'CL-': 'CL', 'K+': 'K'}

def iyon_adlarini_duzelt(gro_girdi, gro_cikti):
    """GRO dosyasindaki Martini 2 iyon adlarini Martini 3 adlarina cevirir."""
    satirlar = open(gro_girdi).read().splitlines()
    n = int(satirlar[1])
    yeni = satirlar[:2]
    degisen = 0
    for s in satirlar[2:2+n]:
        rn, an = s[5:10].strip(), s[10:15].strip()
        if rn in ESLEME or an in ESLEME:
            rn2 = ESLEME.get(rn, rn)
            an2 = ESLEME.get(an, an)
            s = s[:5] + f'{rn2:<5}' + f'{an2:>5}' + s[15:]
            degisen += 1
        yeni.append(s)
    yeni += satirlar[2+n:]          # kutu vektoru satiri
    open(gro_cikti, 'w').write('\n'.join(yeni) + '\n')
    return degisen

n_degisen = iyon_adlarini_duzelt('sistem_ham.gro', 'sistem.gro')
print(f'{n_degisen} iyon parcaciginin adi duzeltildi (NA+ -> NA, CL- -> CL)')

# Dogrulama: dosyada artik NA+/CL- kalmamali
kalan = sum(1 for s in open('sistem.gro').read().splitlines()[2:-1]
            if s[5:10].strip() in ESLEME)
print('Duzeltilmemis kalan iyon:', kalan)


---
## 11. Topolojinin tamamlanması

`insane` bir topoloji dosyası üretmekte, ancak `#include` yönergeleri eksik
olmaktadır; araç proteinin topolojisini tanımamaktadır. Ayrıca iyon adları
burada da düzeltilmelidir.

Yönerge sırası önemlidir: önce genel kuvvet alanı tanımları, ardından molekül
topolojileri yer almalıdır.


In [ ]:
prot_itp = sorted(glob.glob('molecule_*.itp'))[0]

# insane tarafindan uretilen [ molecules ] bolumu
ham = open('sistem_insane.top').read()
mols = [m.strip() for m in ham.split('[ molecules ]')[1].strip().splitlines()
        if m.strip() and not m.strip().startswith(';')]

# Protein molekul adinin martinize2 ciktisiyla eslestirilmesi
prot_ad = None
satirlar = open(prot_itp).read().splitlines()
for i, l in enumerate(satirlar):
    if l.strip().startswith('[ moleculetype ]'):
        for l2 in satirlar[i+1:]:
            if l2.strip() and not l2.strip().startswith(';'):
                prot_ad = l2.split()[0]
                break
        break
print('Protein molekul adi:', prot_ad)

duzeltilmis = []
for m in mols:
    ad, sayi = m.split()[0], m.split()[1]
    if ad.lower().startswith('protein'):
        ad = prot_ad
    ad = ESLEME.get(ad, ad)          # NA+ -> NA, CL- -> CL
    duzeltilmis.append(f'{ad:<12} {sayi}')

top = f'''; Biyofizik 2026 Kursu - Oturum 4
; AT2R (6JOD A zinciri) - Martini 3 - POPC membran

#include "martini_v3.0.0.itp"
#include "martini_v3.0.0_solvents_v1.itp"
#include "martini_v3.0.0_ions_v1.itp"
#include "martini_v3.0.0_phospholipids_v1.itp"
#include "{prot_itp}"

[ system ]
AT2R in POPC membrane (Martini 3)

[ molecules ]
''' + '\n'.join(duzeltilmis) + '\n'

open('sistem.top', 'w').write(top)
print()
print(top)

kaydet(['sistem.gro', 'sistem.top', 'sistem_insane.top'], D_CIKTI)


---
## 12. `gmx grompp` ile doğrulama

Bu adımda simülasyon yürütülmemektedir. Koordinat, topoloji ve parametre
dosyalarının birbiriyle tutarlılığı sınanmaktadır.

`.tpr` dosyasının üretilebilmesi, hazırlanan girdinin geçerli olduğunu
göstermektedir. Oturumun hedefi bu doğrulamanın sağlanmasıdır.


In [ ]:
mdp = '''; Martini 3 - enerji minimizasyonu
integrator               = steep
nsteps                   = 500
emtol                    = 100
emstep                   = 0.01

nstlist                  = 20
cutoff-scheme            = Verlet
verlet-buffer-tolerance  = 0.005

coulombtype              = reaction-field
rcoulomb                 = 1.1
epsilon_r                = 15
vdw_type                 = cutoff
vdw-modifier             = Potential-shift-verlet
rvdw                     = 1.1
'''
open('minimization.mdp', 'w').write(mdp)

!gmx grompp -f minimization.mdp -c sistem.gro -p sistem.top -o em.tpr -maxwarn 5 2>&1 | tail -25


In [ ]:
if os.path.exists('em.tpr'):
    print('BASARILI: em.tpr uretildi ({:,} bayt)'.format(os.path.getsize('em.tpr')))
    print('Girdi hazirligi tamamlanmistir; bu dosya ile simulasyon yurutulebilir.')
else:
    print('em.tpr uretilemedi. Yukaridaki hata iletisi incelenmelidir.')
print()
kaydet(['minimization.mdp', 'em.tpr'], D_CIKTI)


Sabah oturumunda grafik arayüz ile gerçekleştirilen işlem, bu bölümde komut
satırı araçlarıyla ve kaba-taneli çözünürlükte tamamlanmıştır. Komut satırı
yaklaşımının belirleyici üstünlüğü, işlemin tekrarlanabilir ve raporlanabilir
olmasıdır.


### İsteğe bağlı: kısa enerji minimizasyonu

Süre elverdiği takdirde kısa bir minimizasyon çalıştırılarak sistemin kararlı
olduğu doğrulanabilir.


In [ ]:
# !gmx mdrun -deffnm em -nsteps 200 -v 2>&1 | tail -15


---
## 13. Sistemin analizi ve görselleştirilmesi

Kurulan sistemin doğru olup olmadığı görsel olarak denetlenmelidir.
Beklenen görünüm:

- Lipitler kutunun ortasında iki yaprakçıklı dar bir bant oluşturmalıdır
- Çözücü membranın iki yanında toplanmalı, hidrofobik bölgede bulunmamalıdır
- Protein membranı kat etmeli, her iki yana taşmalıdır

### Etkileşimli görünüm

Mavi: protein · Turuncu: lipit · Kırmızı: iyon. Su gizlenmiştir.
Görünümü döndürerek proteinin membran içindeki konumunu inceleyiniz.


In [ ]:
yapi_goster('sistem.gro', stil='sistem', cozucu_gizle=True)


### Yandan kesit görünümü

Kutunun ortasından alınan 2 nm kalınlığındaki bir dilim, membranın yapısını
ve proteinin yerleşimini net biçimde göstermektedir.


In [ ]:
kesit_ciz('sistem.gro', os.path.join(D_GORSEL, '03_sistem_kesit.png'),
          'Adim 3 - AT2R / POPC membran sistemi (yandan kesit)',
          dilim=2.0, eksen=('x', 'z'))


### Üstten görünüm

Membran düzlemine dik bakış; proteinin lipitler arasındaki yerleşimi ve
çevresinde yeterli lipit bulunup bulunmadığı denetlenebilir.


In [ ]:
kesit_ciz('sistem.gro', os.path.join(D_GORSEL, '04_sistem_ustten.png'),
          'Adim 4 - membran duzlemi (ustten gorunum)',
          dilim=2.0, eksen=('x', 'y'))


### z ekseni boyunca bileşen dağılımı

Bu çizim, membranın gerçekten oluşup oluşmadığının **niceliksel** denetimidir.
Lipit eğrisi dar ve tek tepeli, çözücü eğrisi ise membran bölgesinde sıfıra
yakın olmalıdır.


In [ ]:
def z_profili(dosya, png, baslik):
    res, ad, xyz = koordinat_oku(dosya)
    z = xyz[:, 2]
    maskeler = bilesen_maskeleri(res)
    kenar = np.linspace(z.min(), z.max(), 120)
    plt.figure(figsize=(9, 4.5))
    for ad_g, m in maskeler.items():
        if m.sum() == 0:
            continue
        plt.hist(z[m], bins=kenar, histtype='step', lw=1.7,
                 color=RENK[ad_g], label=f'{ad_g} (n={m.sum():,})')
    plt.xlabel('z (nm)'); plt.ylabel('Parcacik sayisi')
    plt.title(baslik); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(png, dpi=150); plt.show()
    print('Kaydedildi:', png)

z_profili('sistem.gro', os.path.join(D_GORSEL, '05_z_profili.png'),
          'Adim 5 - z ekseni boyunca bilesen dagilimi')


> **Çözücü eğrisindeki dişli görünüm hakkında.** Martini suyu düzenli bir
> ızgara üzerine yerleştirildiğinden histogramda periyodik tepeler oluşmaktadır.
> Bu bir kurulum hatası değildir; enerji minimizasyonu ve dengeleme sonrasında
> dağılım düzgünleşmektedir. Önemli olan, çözücünün membran bölgesinde
> **bulunmamasıdır**.


### Membran kalınlığı ve lipit başına alan

Kurulan membranın deneysel değerlerle uyumlu olup olmadığı denetlenmektedir.
POPC için beklenen değerler: kalınlık yaklaşık 3.8–4.0 nm, lipit başına alan
yaklaşık 0.64 nm².


In [ ]:
res, ad, xyz = koordinat_oku('sistem.gro')

# Fosfat (PO4) merkezleri iki yapracigin konumunu verir
po4 = (np.isin(res, list(LIPIT))) & (ad == 'PO4')
z_po4 = xyz[po4, 2]
orta = z_po4.mean()
ust, alt = z_po4[z_po4 > orta], z_po4[z_po4 < orta]
kalinlik = ust.mean() - alt.mean()

# Kutu boyutlari (GRO son satiri)
kutu = [float(v) for v in open('sistem.gro').read().splitlines()[-1].split()[:3]]
alan_lipit = kutu[0] * kutu[1] / max(len(ust), 1)

print(f'Kutu boyutlari      : {kutu[0]:.2f} x {kutu[1]:.2f} x {kutu[2]:.2f} nm')
print(f'Lipit sayisi        : ust {len(ust)}, alt {len(alt)}')
print(f'Membran kalinligi   : {kalinlik:.2f} nm   (POPC beklenen ~3.8-4.0)')
print(f'Lipit basina alan   : {alan_lipit:.3f} nm2  (POPC beklenen ~0.64)')
print()
print('Not: bu degerler dengeleme oncesi kurulum degerleridir; kesin')
print('karsilastirma dengelenmis bir trajektori uzerinden yapilmalidir.')


### Sistem bileşimi


In [ ]:
res, ad, xyz = koordinat_oku('sistem.gro')
maskeler = bilesen_maskeleri(res)
sayim = {k: int(v.sum()) for k, v in maskeler.items()}
toplam = sum(sayim.values())

plt.figure(figsize=(7, 4))
plt.bar(list(sayim), list(sayim.values()),
        color=[RENK[k] for k in sayim])
plt.ylabel('Parcacik sayisi'); plt.yscale('log')
plt.title('Sistem bilesimi')
for i, (k, v) in enumerate(sayim.items()):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
png = os.path.join(D_GORSEL, '06_sistem_bilesimi.png')
plt.savefig(png, dpi=150); plt.show()
print('Kaydedildi:', png)
print()
for k, v in sayim.items():
    print(f'  {k:<8}: {v:>8,}  (%{100*v/toplam:.1f})')


---
## 14. Drive klasörünün özeti

Önceki adımlarda dosyalar kademeli olarak kaydedilmiştir. Aşağıdaki hücre
klasör içeriğini listelemektedir.


In [ ]:
print('Google Drive icerigi:', OTURUM.replace('/content/drive/MyDrive', "Drive'im"))
print()
toplam = 0
for alt in ('girdi', 'cikti', 'gorseller'):
    yol = os.path.join(OTURUM, alt)
    dosyalar = sorted(os.listdir(yol))
    print(f'{alt}/ ({len(dosyalar)} dosya)')
    for d in dosyalar:
        boyut = os.path.getsize(os.path.join(yol, d))
        toplam += boyut
        print(f'    {d:<32} {boyut:>10,} bayt')
    print()
print(f'Toplam: {toplam/1e6:.1f} MB')


### İsteğe bağlı: bilgisayara indirme

Dosyalar Drive'da saklandığından bu adım gerekli değildir.


In [ ]:
# import shutil
# from google.colab import files
# arsiv = shutil.make_archive('/content/oturum4_ciktilar', 'zip', OTURUM)
# files.download(arsiv)


---
## Sık karşılaşılan hata iletileri

| Hata iletisi | Nedeni | Çözümü |
|---|---|---|
| `DSSPError: Expected record CRYST1 but found ATOM` | `mkdssp` 4.x sürümü `CRYST1` kaydı beklemektedir | 6. bölümdeki `-ss` yöntemi kullanılmaktadır; `-dssp` kullanılmamalıdır |
| `No such moleculetype NA+` | `insane` Martini 2 iyon adları üretmektedir | 10. bölümdeki ad dönüştürme adımı çalıştırılmalıdır |
| `Atomtype X not found` | Kuvvet alanı parametre dosyası eksik | 8. bölümdeki indirmeler denetlenmelidir |
| `number of coordinates does not match topology` | `[ molecules ]` sayıları koordinat dosyasıyla uyuşmuyor | 11. bölüm yeniden çalıştırılmalıdır |
| `Unknown molecule type Protein` | Protein adı topoloji dosyalarında farklı | 11. bölüm bu düzeltmeyi otomatik yapmaktadır |
| `System has non-zero total charge` | Yuvarlama kaynaklı; olağandır | `-maxwarn` ile geçilebilir |
| `MessageError: credential propagation was unsuccessful` | Drive bağlama izni verilmedi | 1. bölüm yeniden çalıştırılıp izin verilmelidir |
| py3Dmol görünümü boş | Tarayıcı JavaScript'i engelliyor olabilir | Hücre yeniden çalıştırılmalı; statik çizimler her durumda üretilmektedir |

---

## Kaynaklar

- [Martini Protein Model — Using Martinize2](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsI/Tut1.html)
- [Modeling Complex Lipid Membranes — INSANE](https://cgmartini.nl/docs/tutorials/Martini3/LipidsII/)
- [Notes and Limitations](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsI/Tut4.html)
- Kuvvet alanı dosyaları: [marrink-lab/martini-forcefields](https://github.com/marrink-lab/martini-forcefields)
